# Phase 1 — Bayesian Foundations

Companion notebook to `notes/phase1-bayes-foundations.md`. We:

1. Build a budget-informed Normal prior with `src.priors`.
2. Walk through the Beta-Binomial coin example.
3. Preview the Normal-Normal one-month update for the FYF reference parameters.
4. Visualise prior, likelihood, and posterior on the same axis.

All numerics here cross-check the worked examples in the theory notes.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from src.priors import (
    normal_prior_from_budget,
    gamma_prior_from_rate,
    beta_prior_from_proportion,
    prior_summary,
)

rng = np.random.default_rng(seed=20260509)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## 1. Budget plan ⟶ Normal prior

Reference scenario from `docs/model-design.md`: 50-person team,
average gross salary R$ 12K, benefit multiplier 1.75 ⟹ planned monthly
cost R$ 1,050,000. Planner is 90% confident the truth is within ±15%.

Solve for $\sigma_0$ via $\mu_0 \pm z_{0.95} \sigma_0 = \mu_0(1 \pm 0.15)$,
with $z_{0.95} \approx 1.6449$.

In [ ]:
prior = normal_prior_from_budget(
    plan_value=1_050_000,
    confidence_pct=0.90,
    interval_width=0.15,
)
summary = prior_summary(prior, credible_level=0.95)

print(f"family   = {prior.family}")
print(f"params   = {prior.params}")
print(f"mean     = R$ {summary['mean']:,.0f}")
print(f"std      = R$ {summary['std']:,.0f}")
lo, hi = summary['credible_interval']
print(f"95% CI   = [R$ {lo:,.0f}, R$ {hi:,.0f}]")

## 2. Beta-Binomial coin example

Prior $\theta \sim \text{Beta}(2,2)$ (mean 0.5, mildly informative).
Observe $n=10$, $x=7$ heads. Posterior is $\text{Beta}(9, 5)$, mean
$9/14 \approx 0.643$.

In [ ]:
alpha0, beta0 = 2, 2
n_flips, n_heads = 10, 7
alpha1 = alpha0 + n_heads
beta1 = beta0 + n_flips - n_heads

prior_beta = stats.beta(alpha0, beta0)
post_beta = stats.beta(alpha1, beta1)
grid = np.linspace(0, 1, 400)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grid, prior_beta.pdf(grid), label=f"Prior Beta({alpha0},{beta0})", color="steelblue")
ax.plot(grid, post_beta.pdf(grid), label=f"Posterior Beta({alpha1},{beta1})", color="crimson")
ax.axvline(0.5, color="steelblue", ls=":", alpha=0.6, label="prior mean = 0.5")
ax.axvline(n_heads / n_flips, color="black", ls="--", alpha=0.5, label=f"empirical = {n_heads}/{n_flips}")
ax.axvline(alpha1 / (alpha1 + beta1), color="crimson", ls=":", alpha=0.6, label=f"posterior mean = {alpha1/(alpha1+beta1):.3f}")
ax.set_xlabel(r"$\theta$ (probability of heads)")
ax.set_ylabel("density")
ax.set_title("Beta-Binomial: prior + 10 flips → posterior")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 3. Normal-Normal one-month update (preview)

Reference parameters from `docs/model-design.md`:
$\mu_0 = 1{,}050{,}000$, $\sigma_0 = 150{,}000$, $\sigma = 80{,}000$.
Observe $x_1 = 1{,}120{,}000$. Closed form (full derivation in Phase 2):

$$
\sigma_1^2 = \frac{\sigma^2 \sigma_0^2}{\sigma^2 + \sigma_0^2},
\qquad
\mu_1 = \frac{\sigma^2 \mu_0 + \sigma_0^2 x_1}{\sigma^2 + \sigma_0^2}.
$$

In [ ]:
mu0, sigma0 = 1_050_000.0, 150_000.0
sigma = 80_000.0
x1 = 1_120_000.0

var0 = sigma0 ** 2
varL = sigma ** 2
var1 = (varL * var0) / (varL + var0)
sigma1 = np.sqrt(var1)
mu1 = (varL * mu0 + var0 * x1) / (varL + var0)

print(f"prior      μ0 = {mu0:>12,.0f},  σ0 = {sigma0:>10,.0f}")
print(f"data        x = {x1:>12,.0f},  σ  = {sigma:>10,.0f}")
print(f"posterior  μ1 = {mu1:>12,.0f},  σ1 = {sigma1:>10,.0f}")
print(f"shift Δμ      = {mu1 - mu0:+,.0f}")
print(f"shrinkage     σ1/σ0 = {sigma1 / sigma0:.3f}")

In [ ]:
grid = np.linspace(700_000, 1_400_000, 600)
prior_n = stats.norm(loc=mu0, scale=sigma0)
lik_n = stats.norm(loc=x1, scale=sigma)         # likelihood viewed as fn of θ around x1
post_n = stats.norm(loc=mu1, scale=sigma1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(grid, prior_n.pdf(grid), label=f"Prior  N({mu0:,.0f}, {sigma0:,.0f}²)", color="steelblue")
ax.plot(grid, lik_n.pdf(grid),   label=f"Likelihood (kernel)  N({x1:,.0f}, {sigma:,.0f}²)", color="goldenrod", ls="--")
ax.plot(grid, post_n.pdf(grid),  label=f"Posterior  N({mu1:,.0f}, {sigma1:,.0f}²)", color="crimson")
ax.set_xlabel(r"$\theta$ — mean monthly cost (R\$)")
ax.set_ylabel("density")
ax.set_title("Normal-Normal: one month of data tightens and shifts the prior")
ax.legend(fontsize=9)
ax.ticklabel_format(style="plain", axis="x")
fig.tight_layout()
plt.show()

## 4. Cross-check Gamma and Beta priors

Sanity-check the helpers used elsewhere in the article.

In [ ]:
incidents_prior = gamma_prior_from_rate(expected_rate=3.0, confidence=1.0)
overtime_prior = beta_prior_from_proportion(expected_prop=0.20, sample_size_equiv=10)

for label, p in [("incidents (Gamma)", incidents_prior), ("overtime (Beta)", overtime_prior)]:
    s = prior_summary(p, credible_level=0.95)
    lo, hi = s["credible_interval"]
    print(f"{label}: mean={s['mean']:.3f}, std={s['std']:.3f}, 95% CI=[{lo:.3f}, {hi:.3f}]")

---

**Next phase.** `notes/phase2-conjugate-families.md` derives the
Normal-Normal posterior from scratch (kernel matching, completing the
square in the exponent) and generalises to Normal-Inverse-Gamma,
Gamma-Poisson, and Beta-Binomial. The closed forms used in §3 above
will then be results, not previews.